In [ ]:
import pandas as pd
import requests

# Fetch the data.
df = pd.read_csv("https://ourworldindata.org/grapher/gdp-per-capita-maddison-project-database.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

# Fetch the metadata
metadata = requests.get("https://ourworldindata.org/grapher/gdp-per-capita-maddison-project-database.metadata.json?v=1&csvType=full&useColumnShortNames=true").json()

In [ ]:
df.to_csv("ourworldindata.csv",index=False)

In [ ]:
metadata

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Your existing data loading code
year = np.arange(1860,2030,10)
selectco = {'FRA':'blue', 'DEU':"green", 'CHE':"green", 'GBR':'purple', 'USA':'purple','RUS':"red", 'JPN':'red'}
linestyles = {'USA':":", 'CHE':":", 'JPN':':'}

df = pd.read_csv("ourworldindata.csv")
df = df.loc[df["code"].isin(selectco.keys()) & (df["year"]>=1860),["code","year","gdp_per_capita"]]

# First graph - Growth rates (your original code)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)  # 1 row, 2 columns, first subplot
df['gdp_pct_change'] = np.log(df['gdp_per_capita'])
df['gdp_pct_change'] = df['gdp_pct_change'].diff()/df['year'].diff()
df['gdp_pct_change'] = df['gdp_pct_change'].where(df['code'] == df['code'].shift(1))

for c in selectco.keys():
    cdf = df.loc[df["code"]==c]
    plt.plot(cdf["year"], cdf["gdp_pct_change"].expanding().mean(), 
             label=c, color=selectco[c], linestyle=linestyles.get(c,"-"))
plt.grid(True, alpha=0.3)    
plt.xlabel("Year")
plt.ylabel("Annualized Mean Log Growth")
plt.title("GDP per Capita Growth Rates")
plt.ylim(bottom=0)
plt.legend(loc='upper right')

# Second graph - GDP per capita levels (log scale)
plt.subplot(1, 2, 2)  # 1 row, 2 columns, second subplot

for c in selectco.keys():
    cdf = df.loc[df["code"]==c].copy()
    # Use 1860 as base year = 100 for relative comparison
    cdf['normalized_gdp'] = (cdf['gdp_per_capita']) * 100
    plt.semilogy(cdf["year"], cdf['normalized_gdp'], 
                 label=c, color=selectco[c], linestyle=linestyles.get(c,"-"))

plt.xlabel("Year")
plt.ylabel("Real GDP per Capita (log scale)")
plt.title("GDP per Capita Levels (Log Scale)")
plt.grid(True, alpha=0.3)
plt.legend(loc='upper left')
plt.tight_layout()  # Adjust spacing between subplots
plt.show()

In [ ]:
df